# Level 4: compare the players

The entity is now the player: who plays the most, who wins, against which race, against whom,
and how they play: the workers and the income at 8:00 against the field, the commands per minute,
the openings and the favorite units. On a snapshot of your own replays the player with the most
games is you, and the comparisons are against your opponents.

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two.

Previous: [Level 3](./Level_3_Trends.ipynb). Next: [Level 5](./Level_5_Maps.ipynb).

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

# They read every unit_born file, so they are collected once and reused.
players = s2.players(IPC_DIR).collect().lazy()
games_1v1 = s2.one_vs_one(IPC_DIR).collect().lazy()
games = s2.games(IPC_DIR).collect().lazy()
# The players compared are the ones with at least this many 1v1 games.
MIN_GAMES = s2.at_least(30)

## The leaderboard

Games, win rate and main race of every player with enough 1v1 games. The main race is the race
with the most games, a tie goes to the first letter: `rank().over()` in polars, `row_number()
OVER` with `QUALIFY` in SQL (DuckDB does not combine `QUALIFY` with `GROUP BY ALL` yet, the
columns are named).

The player is the name in the replay without the clan tag, and a name is not a person: a team
name glued to the name makes another player (`Clem`, `LiquidClem`, `clem`), and two people can
share a name. `player_toon_id` of the details identifies the account, per region:
`examples/duckdb_examples.py same_person` lists the names of each one. Merge the names you know
before reading a ranking of real players. The win rate of a player with 30 games has a margin of about 9
points, keep that in mind when reading the differences.

In [2]:
main_race = (
    games_1v1.group_by("player", "race")
    .agg(pl.len().alias("n"))
    .sort("player", "n", "race", descending=[False, True, False])
    .filter(pl.col("n").rank("ordinal", descending=True).over("player") == 1)
    .select("player", pl.col("race").alias("main_race"))
)
leaderboard = (
    games_1v1.group_by("player")
    .agg(
        pl.len().alias("games"),
        (pl.col("player_result") == "Win").mean().alias("win_rate"),
        pl.col("ext_datetime").min().dt.date().alias("first_game"),
        pl.col("ext_datetime").max().dt.date().alias("last_game"),
    )
    .filter(pl.col("games") >= MIN_GAMES)
    .join(main_race, on="player")
    .sort("games", "player", descending=[True, False])
    .collect()
)
# The players of the head to head, the economy and the openings below: the most played ones.
TOP = leaderboard.head(15)["player"].to_list()
leaderboard

player,games,win_rate,first_game,last_game,main_race
str,u32,f64,date,date,str
"""Doombringer""",3065,0.493638,2019-08-13,2023-09-06,"""Z"""
"""Reynor""",1005,0.627861,2020-07-31,2026-08-21,"""Z"""
"""Serral""",938,0.726013,2020-08-02,2026-08-23,"""Z"""
"""ShoWTimE""",750,0.537333,2020-08-02,2026-08-14,"""P"""
"""HeroMarine""",736,0.58288,2020-07-28,2026-08-21,"""T"""
"""ByuN""",704,0.525568,2020-10-20,2026-08-14,"""T"""
"""Lambo""",673,0.539376,2020-07-31,2026-08-13,"""Z"""
"""LiquidClem""",661,0.644478,2020-07-31,2026-05-31,"""T"""
"""Astrea""",644,0.554348,2020-07-31,2025-09-05,"""P"""


In [3]:
sql = con.sql(f"""
    WITH main_race AS (
        SELECT player, race AS main_race, count(*) AS n
        FROM games_1v1
        GROUP BY player, race
        QUALIFY row_number() OVER (PARTITION BY player ORDER BY n DESC, race) = 1
    )
    SELECT
        g.player,
        count(*) AS games,
        avg((g.player_result = 'Win')::INTEGER) AS win_rate,
        min(g.ext_datetime)::DATE AS first_game,
        max(g.ext_datetime)::DATE AS last_game,
        any_value(m.main_race) AS main_race
    FROM games_1v1 g
    JOIN main_race m USING (player)
    GROUP BY g.player
    HAVING count(*) >= {MIN_GAMES}
""")
s2.same(leaderboard, sql)

'same 218 rows, 6 columns'

In [4]:
fig = px.bar(
    leaderboard.head(30).sort("win_rate", descending=True), x="player", y="win_rate", hover_data=["games", "main_race"],
    title=f"Win rate of the 30 most played players with at least {MIN_GAMES} games", labels={"win_rate": "win rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.add_hline(y=0.5, line_color="#898781", line_width=1)
viz.show(fig, "level_4_win_rate")

## Against each race

The win rate of the 15 most played players (`TOP`) against each race, from the long table (one
row per player and opponent race) pivoted for reading. DuckDB's `PIVOT` does the same on the SQL side.

In [5]:
vs_race = (
    games_1v1.filter(pl.col("player").is_in(TOP) & (pl.col("opponent_race") != "?"))
    .group_by("player", "opponent_race")
    .agg(pl.len().alias("games"), (pl.col("player_result") == "Win").mean().alias("win_rate"))
    .sort("player", "opponent_race")
    .collect()
)
vs_race.pivot(on="opponent_race", index="player", values=["games", "win_rate"]).sort("player")

player,games_P,games_T,games_Z,win_rate_P,win_rate_T,win_rate_Z
str,u32,u32,u32,f64,f64,f64
"""Astrea""",257,216,171,0.587549,0.550926,0.508772
"""ByuN""",255,194,255,0.576471,0.536082,0.466667
"""Clem""",168,125,191,0.761905,0.584,0.591623
"""Doombringer""",940,1028,1097,0.488298,0.524319,0.469462
"""HeroMarine""",325,191,220,0.661538,0.528796,0.513636
"""Lambo""",279,242,152,0.655914,0.458678,0.453947
"""LiquidClem""",252,137,272,0.678571,0.583942,0.643382
"""MaxPax""",159,194,167,0.666667,0.515464,0.48503
"""Reynor""",361,366,278,0.6759,0.587432,0.618705


In [6]:
sql = con.sql("""
    SELECT player, opponent_race, count(*) AS games, avg((player_result = 'Win')::INTEGER) AS win_rate
    FROM games_1v1
    WHERE player IN (SELECT unnest(?::VARCHAR[])) AND opponent_race <> '?'
    GROUP BY ALL
""", params=[TOP])
print(s2.same(vs_race, sql))
con.sql("PIVOT sql ON opponent_race USING round(avg(win_rate), 3) GROUP BY player ORDER BY player")

same 45 rows, 4 columns


┌─────────────┬────────┬────────┬────────┐
│   player    │   P    │   T    │   Z    │
│   varchar   │ double │ double │ double │
├─────────────┼────────┼────────┼────────┤
│ Astrea      │  0.588 │  0.551 │  0.509 │
│ ByuN        │  0.576 │  0.536 │  0.467 │
│ Clem        │  0.762 │  0.584 │  0.592 │
│ Doombringer │  0.488 │  0.524 │  0.469 │
│ HeroMarine  │  0.662 │  0.529 │  0.514 │
│ Lambo       │  0.656 │  0.459 │  0.454 │
│ LiquidClem  │  0.679 │  0.584 │  0.643 │
│ MaxPax      │  0.667 │  0.515 │  0.485 │
│ Reynor      │  0.676 │  0.587 │  0.619 │
│ SKillous    │  0.571 │  0.493 │  0.536 │
│ Scarlett    │  0.574 │  0.559 │  0.607 │
│ Serral      │  0.806 │  0.709 │  0.662 │
│ ShoWTimE    │  0.621 │  0.503 │  0.486 │
│ Solar       │  0.584 │  0.553 │  0.528 │
│ herO        │  0.703 │  0.579 │  0.565 │
└─────────────┴────────┴────────┴────────┘
  15 rows                      4 columns

## Head to head

The games between the players of the leaderboard: the wins of the player in the row against the
player in the column. The matrix is read along the row, the darker the more wins.

In [7]:
head_to_head = (
    games_1v1.filter(pl.col("player").is_in(TOP) & pl.col("opponent").is_in(TOP))
    .group_by("player", "opponent")
    .agg(pl.len().alias("games"), (pl.col("player_result") == "Win").sum().alias("wins"))
    .sort("player", "opponent")
    .collect()
)
head_to_head.sort("games", descending=True).head(8)

player,opponent,games,wins
str,str,u32,u32
"""Astrea""","""Scarlett""",69,37
"""Scarlett""","""Astrea""",69,32
"""Reynor""","""Serral""",68,30
"""Serral""","""Reynor""",68,38
"""LiquidClem""","""Serral""",59,27
"""Serral""","""LiquidClem""",59,32
"""Clem""","""Serral""",55,29
"""Serral""","""Clem""",55,26


In [8]:
sql = con.sql("""
    SELECT player, opponent, count(*) AS games, count(*) FILTER (WHERE player_result = 'Win') AS wins
    FROM games_1v1
    WHERE player IN (SELECT unnest(?::VARCHAR[])) AND opponent IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY ALL
""", params=[TOP, TOP])
s2.same(head_to_head, sql)

'same 174 rows, 4 columns'

In [9]:
import plotly.graph_objects as go

matrix = head_to_head.pivot(on="opponent", index="player", values="wins").fill_null(0)
order = [player for player in TOP if player in matrix.columns]
matrix = matrix.filter(pl.col("player").is_in(order)).select("player", *order)
matrix = matrix.sort(pl.col("player").replace_strict(dict(zip(order, range(len(order))))))
fig = go.Figure(
    go.Heatmap(
        z=matrix.select(order).to_numpy(), x=order, y=matrix["player"].to_list(), xgap=2, ygap=2,
        colorscale=[[0, "#fcfcfb"], [0.001, viz.SEQUENTIAL[0]], [1, viz.SEQUENTIAL[-1]]],
        hovertemplate="%{y} beat %{x} %{z} times<extra></extra>", colorbar=dict(title="wins"),
    )
)
fig.update_layout(title="Head to head: wins of the player in the row against the player in the column", height=max(400, 28 * len(order) + 150), width=max(600, 28 * len(order) + 300), yaxis=dict(autorange="reversed"))
viz.show(fig, "level_4_head_to_head")

## The economy at 8:00, against the field

The median workers and income of each player at 8:00 (the first stats sample of the minute),
compared with the median of all the players of the snapshot: the field. A player above the
field builds the economy faster, which for a Zerg is also what the race does.

In [10]:
at_8 = (
    stats.with_columns((s2.game_seconds() // 60).cast(pl.Int32).alias("minute"))
    .filter(pl.col("minute") == 8)
    .group_by("ext_fs_id", "player_id")
    .agg(
        pl.col("workers_active_count").sort_by("ext_replay_loop").first().alias("workers"),
        (pl.col("minerals_collection_rate") + pl.col("vespene_collection_rate")).sort_by("ext_replay_loop").first().alias("income"),
    )
    .join(players.select("ext_fs_id", "player_id", "player", "race"), on=["ext_fs_id", "player_id"])
)
field = at_8.select(pl.col("workers").median().alias("field_workers"), pl.col("income").median().alias("field_income")).collect()
economy = (
    at_8.filter(pl.col("player").is_in(TOP))
    .group_by("player")
    .agg(pl.len().alias("games_at_8"), pl.col("workers").median().alias("median_workers"), pl.col("income").median().alias("median_income"))
    .join(main_race, on="player")
    .sort("median_income", descending=True)
    .collect()
)
print(f"The field at 8:00: {field['field_workers'][0]:.0f} workers, {field['field_income'][0]:.0f} income")
economy

The field at 8:00: 55 workers, 2866 income


player,games_at_8,median_workers,median_income,main_race
str,u32,f64,f64,str
"""Lambo""",546,72.0,3803.5,"""Z"""
"""Serral""",764,71.0,3787.0,"""Z"""
"""Scarlett""",427,71.0,3773.0,"""Z"""
"""Reynor""",812,70.0,3722.0,"""Z"""
"""Solar""",464,70.0,3714.0,"""Z"""
"""ShoWTimE""",604,65.0,3347.0,"""P"""
"""Clem""",401,61.0,3336.0,"""T"""
"""LiquidClem""",540,60.0,3232.5,"""T"""
"""SKillous""",364,62.0,3218.5,"""P"""


In [11]:
sql = con.sql("""
    WITH at_8 AS (
        SELECT ext_fs_id, player_id, arg_min(workers_active_count, ext_replay_loop) AS workers,
               arg_min(minerals_collection_rate + vespene_collection_rate, ext_replay_loop) AS income
        FROM stats
        WHERE floor(game_seconds(ext_replay_loop) / 60)::INTEGER = 8
        GROUP BY ALL
    ),
    main_race AS (
        SELECT player, race AS main_race, count(*) AS n FROM games_1v1 GROUP BY player, race
        QUALIFY row_number() OVER (PARTITION BY player ORDER BY n DESC, race) = 1
    )
    SELECT p.player, count(*) AS games_at_8, median(a.workers) AS median_workers, median(a.income) AS median_income, any_value(m.main_race) AS main_race
    FROM at_8 a
    JOIN players p USING (ext_fs_id, player_id)
    JOIN main_race m USING (player)
    WHERE p.player IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY p.player
""", params=[TOP])
s2.same(economy, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 15 rows, 5 columns'

In [12]:
fig = px.bar(
    economy, x="median_income", y="player", orientation="h", color="main_race", color_discrete_map=viz.RACE_COLORS,
    category_orders={"main_race": ["P", "T", "Z"]}, hover_data=["median_workers", "games_at_8"],
    title="Median income at 8:00 per player, the gray line is the field", labels={"median_income": "income at 8:00 (minerals + vespene per minute)", "main_race": "race"},
    height=max(400, 26 * economy.height + 150),
)
fig.add_vline(x=field["field_income"][0], line_color="#898781", line_width=2)
fig.update_yaxes(autorange="reversed")
viz.show(fig, "level_4_income_at_8")

## Commands per minute

The ability commands of each player per game minute (the `casts`, see Level 1), the median over
the games: a measure of how much a player does, not the APM of the client.

In [13]:
commands = (
    s2.casts(IPC_DIR)
    .group_by("ext_fs_id", "player_name")
    .agg(pl.len().alias("commands"))
    .join(games.select("ext_fs_id", "minutes"), on="ext_fs_id")
    .join(players.select("ext_fs_id", "player_name", "player"), on=["ext_fs_id", "player_name"])
    .filter(pl.col("player").is_in(TOP))
    .with_columns((pl.col("commands") / pl.col("minutes")).alias("per_minute"))
    .group_by("player")
    .agg(pl.len().alias("games"), pl.col("per_minute").median().alias("commands_per_minute"))
    .sort("commands_per_minute", descending=True)
    .collect()
)
commands

player,games,commands_per_minute
str,u32,f64
"""Clem""",484,59.328744
"""LiquidClem""",661,57.080746
"""HeroMarine""",736,48.870927
"""ByuN""",703,45.987357
"""Reynor""",1005,37.074091
"""Lambo""",673,36.373366
"""Serral""",938,35.093237
"""Solar""",590,34.929387
"""Scarlett""",541,32.932696


In [14]:
sql = con.sql("""
    WITH per_game AS (
        SELECT c.ext_fs_id, c.player_name, count(*) AS commands FROM casts c GROUP BY ALL
    )
    SELECT p.player, count(*) AS games, median(c.commands / g.minutes) AS commands_per_minute
    FROM per_game c
    JOIN games g USING (ext_fs_id)
    JOIN players p USING (ext_fs_id, player_name)
    WHERE p.player IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY p.player
""", params=[TOP])
s2.same(commands, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 15 rows, 3 columns'

## The openings

The first three structures of each player in each 1v1 game, after the starting townhall, joined
into one label (`Pylon > Gateway > Assimilator`): the most played opening of each player and how
often it is played. `head(3)` inside the aggregation keeps the first three of the group in
polars, `QUALIFY row_number() <= 3` in SQL, and `str.join` / `string_agg` make the label.

In [15]:
openings = (
    unit_born.filter((pl.col("ext_replay_loop") > 0) & s2.is_structure())
    .join(games_1v1.select("ext_fs_id", pl.col("player_id").alias("control_player_id"), "player", "player_result"), on=["ext_fs_id", "control_player_id"])
    .filter(pl.col("player").is_in(TOP))
    .group_by("ext_fs_id", "player", "player_result")
    .agg(pl.col("unit_type_name").sort_by("ext_replay_loop", "unit_tag_index").head(3).str.join(" > ").alias("opening"))
    .group_by("player", "opening")
    .agg(pl.len().alias("games"), (pl.col("player_result") == "Win").mean().alias("win_rate"))
    .with_columns((pl.col("games") / pl.col("games").sum().over("player")).alias("share"))
    .sort("player", "games", "opening", descending=[False, True, False])
    .collect()
)
openings.filter(pl.col("games").rank("ordinal", descending=True).over("player") == 1)

player,opening,games,win_rate,share
str,str,u32,f64,f64
"""Astrea""","""Pylon > Assimilator > Gateway""",397,0.559194,0.61646
"""ByuN""","""SupplyDepot > Refinery > Barracks""",414,0.519324,0.588905
"""Clem""","""SupplyDepot > Refinery > Barracks""",342,0.675439,0.706612
"""Doombringer""","""Extractor > SpawningPool > Hatchery""",1264,0.456487,0.435112
"""HeroMarine""","""SupplyDepot > Refinery > Barracks""",409,0.572127,0.555707
"""Lambo""","""Extractor > SpawningPool > Hatchery""",286,0.562937,0.42623
"""LiquidClem""","""SupplyDepot > Refinery > Barracks""",493,0.651116,0.74584
"""MaxPax""","""Pylon > Assimilator > Gateway""",433,0.549654,0.834297
"""Reynor""","""Extractor > Hatchery > SpawningPool""",482,0.591286,0.48008


In [16]:
sql = con.sql(f"""
    WITH structures AS (
        SELECT b.ext_fs_id, p.player, p.player_result, b.unit_type_name, b.ext_replay_loop, b.unit_tag_index
        FROM unit_born b
        JOIN games_1v1 p ON p.ext_fs_id = b.ext_fs_id AND p.player_id = b.control_player_id
        WHERE b.ext_replay_loop > 0 AND b.unit_type_name IN ({s2.sql_list(s2.ALL_STRUCTURES)})
          AND p.player IN (SELECT unnest(?::VARCHAR[]))
        QUALIFY row_number() OVER (PARTITION BY b.ext_fs_id, p.player_id ORDER BY b.ext_replay_loop, b.unit_tag_index) <= 3
    ),
    per_game AS (
        SELECT ext_fs_id, player, player_result, string_agg(unit_type_name, ' > ' ORDER BY ext_replay_loop, unit_tag_index) AS opening
        FROM structures
        GROUP BY ALL
    )
    SELECT player, opening, count(*) AS games, avg((player_result = 'Win')::INTEGER) AS win_rate,
           count(*) / sum(count(*)) OVER (PARTITION BY player) AS share
    FROM per_game
    GROUP BY player, opening
""", params=[TOP])
s2.same(openings, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 189 rows, 5 columns'

## The favorite units

The share of the army units each player makes in the 1v1 games, the top three per player: the
units born with a creator ability (trained, warped or morphed), without the workers and the
Overlords.

In [17]:
NOT_ARMY = [*s2.WORKERS, "Overlord", "Larva", "Egg", "MULE", "Broodling", "BroodlingEscort", "Interceptor", "Changeling", "AdeptPhaseShift", "LocustMP", "AutoTurret", "KD8Charge", "ParasiticBombDummy"]
army = (
    unit_born.filter((pl.col("ext_replay_loop") > 0) & (pl.col("creator_ability_name") != "") & pl.col("creator_ability_name").is_not_null() & ~pl.col("unit_type_name").is_in(NOT_ARMY))
    .join(games_1v1.select("ext_fs_id", pl.col("player_id").alias("control_player_id"), "player"), on=["ext_fs_id", "control_player_id"])
    .filter(pl.col("player").is_in(TOP))
    .group_by("player", pl.col("unit_type_name").alias("unit"))
    .agg(pl.len().alias("units"))
    .with_columns((pl.col("units") / pl.col("units").sum().over("player")).alias("share"))
    .sort("player", "units", "unit", descending=[False, True, False])
    .collect()
)
favorites = army.filter(pl.col("units").rank("ordinal", descending=True).over("player") <= 3)
favorites.group_by("player", maintain_order=True).agg(pl.format("{} ({}%)", "unit", (pl.col("share") * 100).round(0).cast(pl.Int32)).str.join(", ").alias("favorite_units"))

player,favorite_units
str,str
"""Astrea""","""DisruptorPhased (27%), Disruptor (13%), Observer (11%)"""
"""ByuN""","""Marine (63%), Marauder (12%), Medivac (5%)"""
"""Clem""","""Marine (57%), Marauder (9%), WidowMine (8%)"""
"""Doombringer""","""Zergling (64%), Hydralisk (13%), Baneling (8%)"""
"""HeroMarine""","""Marine (60%), Marauder (11%), Medivac (6%)"""
"""Lambo""","""Zergling (70%), Roach (11%), Baneling (8%)"""
"""LiquidClem""","""Marine (58%), Marauder (11%), WidowMine (8%)"""
"""MaxPax""","""DisruptorPhased (22%), VoidRay (11%), Phoenix (10%)"""
"""Reynor""","""Zergling (71%), Roach (9%), Baneling (8%)"""


In [18]:
sql = con.sql(f"""
    SELECT p.player, b.unit_type_name AS unit, count(*) AS units, count(*) / sum(count(*)) OVER (PARTITION BY p.player) AS share
    FROM unit_born b
    JOIN games_1v1 p ON p.ext_fs_id = b.ext_fs_id AND p.player_id = b.control_player_id
    WHERE b.ext_replay_loop > 0 AND b.creator_ability_name <> '' AND b.creator_ability_name IS NOT NULL
      AND b.unit_type_name NOT IN ({s2.sql_list(NOT_ARMY)}) AND p.player IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY p.player, b.unit_type_name
""", params=[TOP])
s2.same(army, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 328 rows, 4 columns'

## What to take from Level 4

- Every comparison between players is the same query as Level 1 or 2 grouped by `player`
  instead of by game or month, plus a minimum of games and the field as the reference.
- The top-per-group questions (main race, favorite opening, favorite units) are windows:
  `rank().over()` and `QUALIFY row_number() OVER`.
- The margin of a win rate on 30 games is wide; the differences in the economy at a fixed minute
  are measured on every game and are far more stable.

Next: [Level 5](./Level_5_Maps.ipynb).